# 🌿 Fine-Tune mT5-small for Ayurvedic Chatbot

This notebook fine-tunes **google/mt5-small** on **Hindi Ayurvedic Q&A samples** using **QLoRA** (4-bit quantization + LoRA adapters).

**Runtime:** Go to `Runtime > Change runtime type > GPU (T4)`

## Step 1: Install Dependencies

In [ ]:
%pip install -q torch transformers datasets accelerate
%pip install -q peft bitsandbytes sentencepiece
print('All dependencies installed!')

## Step 2: Upload Training Data

Upload `train.json` and `val.json` from your `chatbot/data/processed/` folder.

In [ ]:
from google.colab import files
import os

os.makedirs('data', exist_ok=True)

print('Upload train.json:')
uploaded = files.upload()
for name in uploaded:
    os.rename(name, 'data/train.json')

print('\nUpload val.json:')
uploaded = files.upload()
for name in uploaded:
    os.rename(name, 'data/val.json')

## Step 3: Configuration

In [ ]:
import torch

# ── Model ──
BASE_MODEL_NAME = 'google/mt5-small'
OUTPUT_DIR = './mt5_ayurvedic_lora'

# ── LoRA ──
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ['q', 'v']

# ── Training ──
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4  # Effective batch = 16
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
MAX_INPUT_LENGTH = 256
MAX_TARGET_LENGTH = 128
WARMUP_STEPS = 100
WEIGHT_DECAY = 0.01

# ── Device check ──
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 4: Load Model with QLoRA

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load tokenizer
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

# Load full model into GPU (no 4-bit quantization or LoRA)
print('Loading full model to GPU...')
model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL_NAME,
    device_map='auto'
)

print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## Step 5: Load, Format, and Tokenize Data

In [ ]:
import json
from datasets import Dataset

# Load raw JSON
with open('data/train.json', 'r', encoding='utf-8') as f:
    train_data = json.load(f)
with open('data/val.json', 'r', encoding='utf-8') as f:
    val_data = json.load(f)

print(f'Train: {len(train_data)} samples')
print(f'Val: {len(val_data)} samples')

# Format for mT5: extract question/answer pairs
def format_for_mt5(data):
    inputs, targets = [], []
    for item in data:
        question = item.get('question_hi', item.get('question', ''))
        answer = item.get('answer_hi', item.get('answer', ''))
        if not question or not answer:
            continue
        inputs.append(f'\u092a\u094d\u0930\u0936\u094d\u0928: {question}')
        targets.append(answer)
    return {'input_text': inputs, 'target_text': targets}

train_dict = format_for_mt5(train_data)
val_dict = format_for_mt5(val_data)

# Create HuggingFace Datasets
train_dataset = Dataset.from_dict(train_dict)
val_dataset = Dataset.from_dict(val_dict)

print(f'Formatted - Train: {len(train_dataset)} | Val: {len(val_dataset)}')
print(f'\nSample input:  {train_dataset[0]["input_text"][:100]}...')
print(f'Sample target: {train_dataset[0]["target_text"][:100]}...')

# Tokenize - NO padding, NO return_tensors (DataCollator handles both)
def tokenize_function(examples):
    model_inputs = tokenizer(
        examples['input_text'],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=examples['target_text'],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

print('\nTokenizing train set...')
train_tokenized = train_dataset.map(
    tokenize_function, batched=True,
    remove_columns=['input_text', 'target_text'],
    desc='Tokenizing train'
)

print('Tokenizing val set...')
val_tokenized = val_dataset.map(
    tokenize_function, batched=True,
    remove_columns=['input_text', 'target_text'],
    desc='Tokenizing val'
)

# Sanity check
sample = train_tokenized[0]
print(f'\n--- Sanity Check ---')
print(f'Input IDs length:  {len(sample["input_ids"])}')
print(f'Labels length:     {len(sample["labels"])}')
print(f'Non-pad labels:    {sum(1 for x in sample["labels"] if x != 0)}')
print(f'First 10 labels:   {sample["labels"][:10]}')
print(f'Decoded input:     {tokenizer.decode(sample["input_ids"], skip_special_tokens=True)[:80]}...')
print(f'Decoded target:    {tokenizer.decode(sample["labels"], skip_special_tokens=True)[:80]}...')

## Step 6: Prepare Trainer

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

# DataCollator handles padding, tensor conversion, label masking
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
)

# CRITICAL: fp16=False and bf16=False
# mT5 has 250K vocab -> logits overflow in float16 -> nan loss
# The model is already in 4-bit so VRAM is fine without AMP
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    logging_steps=50,
    eval_strategy='steps',
    eval_steps=500,
    save_strategy='steps',
    save_steps=500,
    save_total_limit=2,
    fp16=False,
    bf16=False,
    max_grad_norm=1.0,
    report_to='none',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print(f'Ready to train!')
print(f'  Epochs: {NUM_EPOCHS}')
print(f'  Batch size: {TRAIN_BATCH_SIZE} (effective: {TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS})')
print(f'  Training samples: {len(train_tokenized)}')
print(f'  Validation samples: {len(val_tokenized)}')

## Step 7: Debug - Verify Loss Before Training

Quick check that the model produces a valid (non-zero, non-nan) loss before committing to full training.

In [ ]:
# Quick forward pass to verify loss is valid
batch = next(iter(trainer.get_train_dataloader()))
batch = {k: v.to(model.device) for k, v in batch.items()}

model.train()
with torch.no_grad():
    outputs = model(**batch)

print(f'Loss value:      {outputs.loss.item():.4f}')
print(f'Loss is nan:     {torch.isnan(outputs.loss).item()}')
print(f'Loss is inf:     {torch.isinf(outputs.loss).item()}')
print(f'Logits shape:    {outputs.logits.shape}')
print(f'Logits has nan:  {torch.isnan(outputs.logits).any().item()}')
print(f'Logits has inf:  {torch.isinf(outputs.logits).any().item()}')
print(f'Logits range:    [{outputs.logits.min().item():.4f}, {outputs.logits.max().item():.4f}]')
print(f'Labels non-mask: {(batch["labels"] != -100).sum().item()}')

if not torch.isnan(outputs.loss) and outputs.loss.item() > 0:
    print('\n>>> Loss is VALID - safe to train!')
else:
    print('\n>>> WARNING: Loss is broken - DO NOT train until fixed!')

## Step 8: Train! (~3-4 hours on T4)

**Only run this if the debug cell above shows a valid loss.**

In [ ]:
trainer.train()

## Step 9: Save the Fine-Tuned Model

In [ ]:
# Save the full model + tokenizer
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Full Model saved to: {OUTPUT_DIR}')

# Show saved files
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f'  {f}: {size/1024:.1f} KB')

## Step 10: Test the Model

In [ ]:
# Quick test
model.eval()
test_questions = [
    'प्रश्न: अश्वगंधा के फायदे क्या हैं?',
    'प्रश्न: वात दोष को कैसे संतुलित करें?',
    'प्रश्न: पंचकर्म चिकित्सा क्या है?',
    'प्रश्न: त्रिदोष क्या हैं?',
    'प्रश्न: आयुर्वेद के आठ अंग कौन से हैं?',
]

for q in test_questions:
    inputs = tokenizer(q, max_length=256, truncation=True, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=128, num_beams=4, repetition_penalty=1.2)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = answer.replace('<extra_id_0>', '').strip()
    print(f'Q: {q}')
    print(f'A: {answer}\n')

## Step 11: Backup the Full Model to Google Drive

This will mount your Google Drive and copy the 1.2GB model directory directly into your Drive so you don't have to leave your browser open to download it!

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

drive_folder = '/content/drive/MyDrive/Ayurvedic_mT5_Model'
os.makedirs(drive_folder, exist_ok=True)

print(f'\nCopying {OUTPUT_DIR} to {drive_folder} ... this might take a minute.')

# Use cp to recursively copy the model's safe tensors and tokenizer files
!cp -r {OUTPUT_DIR}/* "{drive_folder}/"

print('\n\u2705 Backup complete! You can close Colab now.')